# Tutorial #2 - GRU Training with Live Metrics

This notebook trains a GRU model on the Jena Climate dataset with per-epoch MLflow metric logging.
The `noted` platform intercepts `mlflow.log_metric()` calls and streams them to the Live Metrics panel in real time.

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import mlflow

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

I0000 00:00:1773935706.785380     222 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773935706.817640     222 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1773935707.495225     222 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1. Data Loading and Cleaning

In [2]:
csv_path = "data/jena_climate_2009_2016.csv"
df = pd.read_csv(csv_path)

# Fix sensor error codes (-9999.0 in wind speed)
df[["wv (m/s)", "max. wv (m/s)"]] = df[["wv (m/s)", "max. wv (m/s)"]].replace(-9999.0, np.nan)

# Parse dates
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")

# Clean
df.columns = df.columns.str.strip()
df = df.drop_duplicates(keep="first")
df = df.sort_values("Date Time").reset_index(drop=True)

print("Raw shape:", df.shape)

Raw shape: (420224, 15)


## 2. Hourly Resampling

In [3]:
all_feature_keys = [
    "p (mbar)", "T (degC)", "Tpot (K)", "Tdew (degC)", "rh (%)",
    "VPmax (mbar)", "VPact (mbar)", "VPdef (mbar)", "sh (g/kg)",
    "H2OC (mmol/mol)", "rho (g/m**3)", "wv (m/s)", "max. wv (m/s)", "wd (deg)"
]

df_hourly = (
    df.set_index("Date Time")[all_feature_keys]
      .resample("1h")
      .mean()
      .reset_index()
)

df_hourly = df_hourly.dropna().reset_index(drop=True)
print("Hourly clean shape:", df_hourly.shape)

Hourly clean shape: (70038, 15)


## 3. Feature Selection and Temporal Split

In [4]:
target_col = "T (degC)"
feature_keys = ["T (degC)", "p (mbar)", "rh (%)", "wv (m/s)", "max. wv (m/s)", "wd (deg)"]

df_work = df_hourly[["Date Time"] + feature_keys].copy()
df_work = df_work.set_index("Date Time")

# 70/15/15 split
n_total = len(df_work)
n_train = int(n_total * 0.70)
n_val   = int(n_total * 0.15)

df_train = df_work.iloc[:n_train]
df_val   = df_work.iloc[n_train:n_train + n_val]
df_test  = df_work.iloc[n_train + n_val:]

print(f"Train: {df_train.shape}, Val: {df_val.shape}, Test: {df_test.shape}")

Train: (49026, 6), Val: (10505, 6), Test: (10507, 6)


## 4. Feature Engineering and Scaling

In [5]:
def add_time_features(df_in):
    df = df_in.copy()
    hour = df.index.hour
    doy = df.index.dayofyear
    df["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    df["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    df["doy_sin"] = np.sin(2 * np.pi * doy / 365.25)
    df["doy_cos"] = np.cos(2 * np.pi * doy / 365.25)
    wd = df["wd (deg)"].astype(float)
    df["wd_sin"] = np.sin(2 * np.pi * wd / 360.0)
    df["wd_cos"] = np.cos(2 * np.pi * wd / 360.0)
    df.drop(columns=["wd (deg)"], inplace=True)
    return df

df_train_fe = add_time_features(df_train)
df_val_fe   = add_time_features(df_val)
df_test_fe  = add_time_features(df_test)

feat_cols = df_train_fe.columns.tolist()
scaler = StandardScaler()
scaler.fit(df_train_fe[feat_cols])

df_train_s = pd.DataFrame(scaler.transform(df_train_fe[feat_cols]), index=df_train_fe.index, columns=feat_cols)
df_val_s   = pd.DataFrame(scaler.transform(df_val_fe[feat_cols]),   index=df_val_fe.index,   columns=feat_cols)
df_test_s  = pd.DataFrame(scaler.transform(df_test_fe[feat_cols]),  index=df_test_fe.index,  columns=feat_cols)

print("Features:", len(feat_cols), feat_cols)

Features: 11 ['T (degC)', 'p (mbar)', 'rh (%)', 'wv (m/s)', 'max. wv (m/s)', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos', 'wd_sin', 'wd_cos']


## 5. Sliding Windows (L=120, H=24)

In [6]:
from numpy.lib.stride_tricks import sliding_window_view

L = 120  # input window (hours)
H = 24   # forecast horizon (hours)

def make_windows(df_scaled, target_name, L, H):
    data = df_scaled.values.astype(np.float32)
    target_idx = df_scaled.columns.get_loc(target_name)
    n_samples = len(data) - L - H
    X = sliding_window_view(data, window_shape=L, axis=0).transpose(0, 2, 1)[:n_samples].copy()
    y = sliding_window_view(data[L:, target_idx], window_shape=H)[:n_samples].copy()
    return X, y

X_train, y_train = make_windows(df_train_s, target_col, L, H)
X_val,   y_val   = make_windows(df_val_s,   target_col, L, H)
X_test,  y_test  = make_windows(df_test_s,  target_col, L, H)

n_features = X_train.shape[2]
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape},  y_test:  {y_test.shape}")

X_train: (48882, 120, 11), y_train: (48882, 24)
X_val:   (10361, 120, 11),   y_val:   (10361, 24)
X_test:  (10363, 120, 11),  y_test:  (10363, 24)


## 6. GRU Model

In [7]:
tf.random.set_seed(42)
np.random.seed(42)

def build_gru_model(L, n_features, H, units1=64, units2=32, n_layers=1,
                    dropout=0.2, l2=0.0, dense_units=0, learning_rate=1e-3,
                    clipnorm=1.0):
    reg = keras.regularizers.l2(l2) if l2 and l2 > 0 else None
    inputs = keras.Input(shape=(L, n_features))
    x = inputs
    x = layers.GRU(units1, return_sequences=(n_layers >= 2),
                   dropout=dropout, recurrent_dropout=0.0,
                   kernel_regularizer=reg)(x)
    if n_layers >= 2:
        x = layers.GRU(units2, return_sequences=False,
                       dropout=dropout, recurrent_dropout=0.0,
                       kernel_regularizer=reg)(x)
    if dense_units and dense_units > 0:
        x = layers.Dense(dense_units, activation="relu", kernel_regularizer=reg)(x)
        x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(H)(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate, clipnorm=clipnorm),
                  loss="mse", metrics=["mae"])
    return model

model = build_gru_model(L, n_features, H, units1=128, units2=64, n_layers=2,
                         dropout=0.2, learning_rate=5e-4)
model.summary()

I0000 00:00:1773935709.441036     222 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 11219 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 120, 11)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ (None, 120, 128)       │        54,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 24)             │         1,560 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 92,952 (363.09 KB)

 Trainable params: 92,952 (363.09 KB)

 Non-trainable params: 0 (0.00 B)

## 7. Training with Live MLflow Metrics

The custom callback calls `mlflow.log_metric()` after every epoch.
The `noted` platform monkey-patches this call to emit real-time events to the Live Metrics panel.

In [8]:
class MLflowEpochLogger(keras.callbacks.Callback):
    """Logs train_loss, val_loss, train_mae, val_mae to MLflow after each epoch."""
    def on_epoch_end(self, epoch, logs=None):
        if logs is None:
            return
        mlflow.log_metric("train_loss", logs.get("loss", 0), step=epoch)
        mlflow.log_metric("val_loss", logs.get("val_loss", 0), step=epoch)
        mlflow.log_metric("train_mae", logs.get("mae", 0), step=epoch)
        mlflow.log_metric("val_mae", logs.get("val_mae", 0), step=epoch)

with mlflow.start_run(run_name="gru_live_metrics_test"):
    mlflow.log_param("model_type", "GRU")
    mlflow.log_param("units1", 128)
    mlflow.log_param("units2", 64)
    mlflow.log_param("n_layers", 2)
    mlflow.log_param("dropout", 0.2)
    mlflow.log_param("learning_rate", 5e-4)
    mlflow.log_param("L", L)
    mlflow.log_param("H", H)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=30,
        batch_size=256,
        verbose=1,
        callbacks=[
            MLflowEpochLogger(),
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
        ]
    )

    # Log model artifact with MLmodel metadata
    import os, shutil, yaml as _yaml
    _model_dir = "/tmp/noted_model"
    if os.path.exists(_model_dir):
        shutil.rmtree(_model_dir)
    os.makedirs(_model_dir)
    model.save(os.path.join(_model_dir, "model.keras"))
    _mlmodel = {
        "flavors": {
            "keras": {
                "keras_version": keras.__version__,
                "save_format": "keras_v3",
                "data": "model.keras",
            },
            "python_function": {
                "loader_module": "mlflow.keras",
                "data": "model.keras",
            },
        },
        "mlflow_version": mlflow.__version__,
        "model_size_bytes": os.path.getsize(os.path.join(_model_dir, "model.keras")),
        "utc_time_created": __import__("datetime").datetime.now(__import__("datetime").timezone.utc).isoformat(),
    }
    with open(os.path.join(_model_dir, "MLmodel"), "w") as _f:
        _yaml.dump(_mlmodel, _f, default_flow_style=False, sort_keys=False)
    mlflow.log_artifacts(_model_dir, artifact_path="model")

Epoch 1/30


I0000 00:00:1773935711.490687     327 cuda_dnn.cc:461] Loaded cuDNN version 92000


  1/191 ━━━━━━━━━━━━━━━━━━━━ 5:09 2s/step - loss: 1.1000 - mae: 0.8542

  5/191 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 1.0615 - mae: 0.8426

  9/191 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 1.0267 - mae: 0.8262

...

181/191 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.1027 - mae: 0.2397

185/191 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.1027 - mae: 0.2397

189/191 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - loss: 0.1027 - mae: 0.2397

191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - loss: 0.1019 - mae: 0.2394 - val_loss: 0.0962 - val_mae: 0.2415 - learning_rate: 2.5000e-04


🏃 View run gru_live_metrics_test at: http://mlflow:5000/#/experiments/3/runs/a295a4c94f3949a49a13ce595cacbc7c
🧪 View experiment at: http://mlflow:5000/#/experiments/3


## 8. Evaluation

In [9]:
def inverse_scale_target(y_scaled, scaler, feature_names, target_col):
    t_idx = feature_names.index(target_col)
    mean = scaler.mean_[t_idx]
    std  = scaler.scale_[t_idx]
    return y_scaled * std + mean

y_pred_scaled = model.predict(X_test, verbose=0)

# Scaled metrics
yt_s = y_test.reshape(-1)
yp_s = y_pred_scaled.reshape(-1)
print(f"TEST (scaled) - MAE: {mean_absolute_error(yt_s, yp_s):.4f}, RMSE: {np.sqrt(mean_squared_error(yt_s, yp_s)):.4f}, R2: {r2_score(yt_s, yp_s):.4f}")

# Metrics in degrees C
y_test_c = inverse_scale_target(y_test, scaler, feat_cols, target_col)
y_pred_c = inverse_scale_target(y_pred_scaled, scaler, feat_cols, target_col)

yt = y_test_c.reshape(-1)
yp = y_pred_c.reshape(-1)
mae_c = mean_absolute_error(yt, yp)
rmse_c = np.sqrt(mean_squared_error(yt, yp))
r2_c = r2_score(yt, yp)
print(f"TEST (C)      - MAE: {mae_c:.4f}, RMSE: {rmse_c:.4f}, R2: {r2_c:.4f}")

# Log final metrics to MLflow too
with mlflow.start_run(mlflow.last_active_run().info.run_id):
    mlflow.log_metric("test_mae_c", mae_c)
    mlflow.log_metric("test_rmse_c", rmse_c)
    mlflow.log_metric("test_r2_c", r2_c)

# Log training history plot as artifact
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['loss'], label='train_loss')
ax[0].plot(history.history['val_loss'], label='val_loss')
ax[0].set_title('Loss')
ax[0].legend()
ax[1].plot(history.history['mae'], label='train_mae')
ax[1].plot(history.history['val_mae'], label='val_mae')
ax[1].set_title('MAE')
ax[1].legend()
fig.tight_layout()
fig.savefig('/tmp/training_curves.png', dpi=100)
plt.close(fig)

with mlflow.start_run(mlflow.last_active_run().info.run_id):
    mlflow.log_artifact('/tmp/training_curves.png')

TEST (scaled) - MAE: 0.2367, RMSE: 0.2988, R2: 0.8907
TEST (C)      - MAE: 2.0460, RMSE: 2.5827, R2: 0.8907


🏃 View run gru_live_metrics_test at: http://mlflow:5000/#/experiments/3/runs/a295a4c94f3949a49a13ce595cacbc7c
🧪 View experiment at: http://mlflow:5000/#/experiments/3


🏃 View run gru_live_metrics_test at: http://mlflow:5000/#/experiments/3/runs/a295a4c94f3949a49a13ce595cacbc7c
🧪 View experiment at: http://mlflow:5000/#/experiments/3


## 9. Baseline Comparison

In [10]:
# Persistence baseline
target_idx = df_test_s.columns.get_loc(target_col)
last_T = X_test[:, -1, target_idx]
y_pred_persist = np.repeat(last_T[:, None], y_test.shape[1], axis=1)

y_persist_c = inverse_scale_target(y_pred_persist, scaler, feat_cols, target_col)
mae_p = mean_absolute_error(y_test_c.reshape(-1), y_persist_c.reshape(-1))
rmse_p = np.sqrt(mean_squared_error(y_test_c.reshape(-1), y_persist_c.reshape(-1)))

print(f"Persistence - MAE: {mae_p:.4f} C, RMSE: {rmse_p:.4f} C")
print(f"GRU         - MAE: {mae_c:.4f} C, RMSE: {rmse_c:.4f} C")
print(f"Improvement - MAE: {(1 - mae_c/mae_p)*100:.1f}%, RMSE: {(1 - rmse_c/rmse_p)*100:.1f}%")

Persistence - MAE: 3.1437 C, RMSE: 4.2542 C
GRU         - MAE: 2.0460 C, RMSE: 2.5827 C
Improvement - MAE: 34.9%, RMSE: 39.3%
